# From Space to Action - Agricultural Drought Early Warning
## Notebook 03: Feature Engineering
**Goal:** Compute all drought-relevant features.

In [ ]:
import os, sys
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/FromSpaceToAction'
DATA_DIR = f'{PROJECT_DIR}/data'
SRC_DIR = f'{PROJECT_DIR}/src'
MODELS_DIR = f'{PROJECT_DIR}/models'
OUTPUTS_DIR = f'{PROJECT_DIR}/outputs'
CONFIG_PATH = f'{PROJECT_DIR}/config/config.yaml'

sys.path.insert(0, SRC_DIR)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm


### Load processed data

In [ ]:
processed_path = f'{DATA_DIR}/processed/processed_data.parquet'
if os.path.exists(processed_path):
    df = pd.read_parquet(processed_path)
    print(f"Loaded processed data shape: {df.shape}")
else:
    print("Processed data not found!")

### Section: Vegetation indices and VCI computation

In [ ]:
df['doy'] = df['dekad'].dt.dayofyear

# Compute VCI (Vegetation Condition Index)
min_max = df.groupby(['lat', 'lon', 'doy'])['ndvi'].agg(['min', 'max']).reset_index()
df = pd.merge(df, min_max, on=['lat', 'lon', 'doy'], how='left')
df['vci'] = 100 * (df['ndvi'] - df['min']) / (df['max'] - df['min'] + 1e-6)
df.drop(columns=['min', 'max'], inplace=True)
print("VCI computed.")

### Section: Rainfall, Soil Moisture, ERA5 features

In [ ]:
# Simulating additional features based on existing ones
df['soil_moisture_decline_rate'] = df.groupby(['lat', 'lon'])['soil_moisture'].diff()
df['temp_anomaly'] = df.get('temp_anomaly', df['temperature'] * 0.1)
print("Additional physical features computed.")

### Section: Temporal features (lags, rolling means, trend)

In [ ]:
df = df.sort_values(by=['lat', 'lon', 'dekad'])
for lag in [1, 2, 4]:
    df[f'vci_lag_{lag}'] = df.groupby(['lat', 'lon'])['vci'].shift(lag)
    df[f'rainfall_30d_anomaly_lag_{lag}'] = df.groupby(['lat', 'lon'])['rainfall_30d_anomaly'].shift(lag)

for window in [3, 6]:
    df[f'vci_roll_{window}'] = df.groupby(['lat', 'lon'])['vci'].transform(lambda x: x.rolling(window, min_periods=1).mean())

print("Temporal features computed.")

### Section: Seasonal encoding & Static features

In [ ]:
df['month_sin'] = np.sin(2 * np.pi * df['dekad'].dt.month / 12)
df['month_cos'] = np.cos(2 * np.pi * df['dekad'].dt.month / 12)

# Mock static features
df['elevation'] = 1500 + np.random.normal(0, 200, len(df))
df['land_cover'] = np.random.choice([10, 20, 30], len(df))
print("Seasonal and static features computed.")

### Feature correlation matrix plot & Summary statistics

In [ ]:
feat_cols = ['vci', 'rainfall_30d_anomaly', 'smap_anomaly', 'temp_anomaly', 'elevation']
corr = df[feat_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Feature Correlation Matrix')
plt.show()

display(df[feat_cols].describe())

### Save feature matrix

In [ ]:
features_path = f'{DATA_DIR}/features/features_v1.parquet'
df.to_parquet(features_path)
print(f"Feature matrix saved to {features_path}")